# SmartVal · Why hidden layers and backpropagation exist

[ML Basics](../02-ml-basics/ml-basics.ipynb) showed how to judge predictions without letting the test examples teach the model. It left a different question open: **what if the model's rule is too simple to express the answer, however honestly we evaluate it?**

Keep SmartVal, but replace noisy rents with four cases you can inspect completely. Two on/off signals describe a case. Flag it when **exactly one signal is on**:

| Signal A | Signal B | Desired flag |
| ---: | ---: | ---: |
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

That rule is **exclusive OR**, or XOR. The signals are abstract teaching inputs, not selected real-estate features. We switch tasks so dataset noise cannot hide the representational problem.

## 0 · What must the model change before it can get this right?

A single weighted score followed by a cutoff divides the input picture with one straight line. XOR puts matching labels on opposite corners. More practice with the same straight-line rule cannot change that geometry.

| Attempt | Complaint | What we'll inspect |
| --- | --- | --- |
| One weighted score | Opposite corners resist a straight split | A finite grid search and the four-point plot |
| Two intermediate features | The rule can work, but how? | Every case's new coordinates and score |
| A manual backward pass | A calculation may be wrong | Four input-weight gradients checked by nudging weights |

An intermediate feature is called a **hidden activation**: a number computed inside the model, before the output. “Hidden” means intermediate, not unknowable. We'll print all of them.

The first cell makes `X` with shape `(4, 2)`: four cases, two signals. `y` has shape `(4, 1)`: one desired output per case. Keep that last axis. Subtracting `(4,)` targets from `(4, 1)` predictions would broadcast into a `(4, 4)` table of mismatched errors.

This is an expressiveness and derivative lab, not a trained XOR model or a held-out benchmark. We set useful weights by hand, then audit a separate one-case backward calculation. There is no optimizer loop in this notebook.

In [ ]:
# -- Imports and the four SmartVal XOR cases ---------------------------------
import matplotlib.pyplot as plt
import numpy as np

# (4, 2): rows are cases, columns are signals, so every layer preserves case identity.
X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
# (4, 1) matches prediction columns; (4,) would accidentally broadcast errors to (4, 4).
y = np.array([[0.], [1.], [1.], [0.]])
print(f"Input contract: X={X.shape} (case, signal), y={y.shape} (case, decision)")


## 1 · Attempt 1: one straight boundary

The score is `w1*A + w2*B + bias`. Each weight says how one signal changes the score; the bias shifts the cutoff for all cases. The code flags scores greater than or equal to zero.

For example, `w1 = w2 = 1`, `bias = -0.5` gives scores `[-0.5, 0.5, 0.5, 1.5]` in table order. It catches both positive cases but also flags `[1, 1]`: three of four correct. This is one member of the grid below, not a learned solution.

**Predict:** can rotating or shifting the line fix that last corner without losing another?

The next cell tries 49 choices for each of two weights and the bias: `49³ = 117,649` lines. It records the best fraction correct. The following plot shows the same four cases in their original coordinates; it does not plot a fitted separating line.

A finite search alone cannot rule out every possible line. The geometric reason is stronger: the midpoint of the two positive corners and the midpoint of the two negative corners are both `[0.5, 0.5]`. An affine score at that midpoint must average either pair's scores. It cannot be nonnegative when averaged from positives and negative when averaged from negatives.

In [ ]:
# -- Exhaust a visible grid of linear boundaries ------------------------------
best = (0.0, None)
for w1 in np.linspace(-3, 3, 49):
    for w2 in np.linspace(-3, 3, 49):
        for bias in np.linspace(-3, 3, 49):
            # (4, 2) @ (2, 1) -> (4, 1); one scalar bias shifts every case's score equally.
            prediction = ((X @ np.array([[w1], [w2]]) + bias) >= 0).astype(float)
            accuracy = float((prediction == y).mean())
            if accuracy > best[0]:
                # (4, 1) -> (4,) only for a readable saved pattern, not for loss comparison.
                best = (accuracy, (w1, w2, bias, prediction.ravel()))

print(f"Best searched linear accuracy: {best[0]:.0%}")
print(f"One best prediction pattern: {best[1][3].astype(int).tolist()}")
print("Complaint: one affine score cannot isolate opposite corners.")


In [ ]:
# -- Static geometry: see why one line is the wrong tool ----------------------
fig, ax = plt.subplots(figsize=(5, 5))
for label, marker in ((0, "o"), (1, "s")):
    # (4, 1) labels -> (4,) row mask; selecting a class gives its two (signal A, B) rows.
    points = X[y.ravel() == label]
    # (2, 2) -> two (2,) coordinate columns, keeping each point's x/y paired for the plot.
    ax.scatter(points[:, 0], points[:, 1], s=160, marker=marker, label=f"target {label}")
# (4, 1) -> (4,) lets zip pair one scalar label with each input row.
for point, label in zip(X, y.ravel()):
    ax.annotate(f" {int(label)}", point)
ax.set(xlabel="signal A", ylabel="signal B", title="XOR keeps equal labels on opposite corners")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("The plot and grid search expose the same limitation.")


## 2 · Attempt 2: change the coordinates before deciding

One line is the wrong tool **in the original picture**. What if we create two useful measurements first?

Start with the total `A+B`. It gives `0, 1, 1, 2`: now the only unwanted high score is “both on.” Add a second measurement `max(A+B-1, 0)`. It fires only for that both-on case. Subtract twice that second measurement and the problem disappears.

The gate `max(value, 0)` is a **ReLU**: retain positive values and replace negative ones with zero. It is not a threshold returning only 0 or 1. Our first hidden feature reaches 2.

| Input | First feature: `max(A+B, 0)` | Second: `max(A+B-1, 0)` | First minus twice second |
| --- | ---: | ---: | ---: |
| `[0, 0]` | 0 | 0 | 0 |
| `[0, 1]` | 1 | 0 | 1 |
| `[1, 0]` | 1 | 0 | 1 |
| `[1, 1]` | 2 | 1 | 0 |

“OR-like” and “AND-like” in the code are mnemonic names for these numerical features, not exact Boolean operators outside these four inputs.

> **Visual to add — Fold the XOR square without relabelling it**
>
> On the left draw the four inputs at their square corners with labels 0, 1, 1, 0. On the right place them at the actual hidden coordinates [0,0], [1,0], [1,0], and [2,1]; use an offset annotation to show the two cases sharing [1,0]. Draw a possible separator h1-2*h2=0.5, labelling it as an illustrative boundary; the code uses score > 0 on these exact 0/1 scores. Preserve each case's colour and name. Do not redraw labels to create separability or imply these hand-chosen features were discovered by training.
>
> **Proposed caption:** “The same cases become easy to separate after a nonlinear change of coordinates.”

Read the implementation as:

```text
X (4,2) -> X @ W_hidden + b_hidden (4,2)
        -> ReLU hidden features (4,2)
        -> hidden @ W_out + b_out (4,1) -> score > 0 -> flags
```

One orientation detail matters. The maths chapter put output recipes in matrix **rows** and multiplied `W @ x`. Here cases are rows, so we multiply `X @ W_hidden`; the hidden recipes are matrix **columns**. Same dot products, transposed storage convention. `b_hidden = [[0, -1]]` supplies one shared offset per hidden column.

`W_out = [[1], [-2]]` combines the two hidden features. On these four cases the scores are exactly 0 or 1, not calibrated probabilities. The side-by-side scatter plots keep each case's label while changing only its coordinates.

In [ ]:
# -- Build a readable hidden representation with fixed weights ----------------
W_hidden = np.array([[1., 1.], [1., 1.]])
b_hidden = np.array([[0., -1.]])
W_out = np.array([[1.], [-2.]])
b_out = np.array([[0.]])

# (4, 2) @ (2, 2) -> (4, 2); b_hidden=(1, 2) broadcasts one shared bias row
# over all four cases, so each hidden unit asks the same question for every case.
hidden_pre = X @ W_hidden + b_hidden
hidden = np.maximum(hidden_pre, 0.0)
# (4, 2) @ (2, 1) -> (4, 1); b_out=(1, 1) shifts each case's single output.
score = hidden @ W_out + b_out
prediction = (score > 0).astype(float)

print("case -> hidden coordinates -> score -> prediction")
# Each score/prediction row is (1,); [0] unwraps that scalar for display,
# without flattening the full (4, 1) case-by-output table.
for x_row, h_row, score_row, pred_row in zip(X, hidden, score, prediction):
    print(f"{x_row.astype(int).tolist()} -> {h_row.tolist()} -> {score_row[0]:.1f} -> {int(pred_row[0])}")
print(f"Hidden representation accuracy: {(prediction == y).mean():.0%}")


In [ ]:
# -- Show that the hidden coordinates make a straight split possible ----------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# For both panels, (4, 2) -> two (4,) coordinate columns; labels (4, 1) -> (4,).
# Keep row order unchanged so each case keeps its color when its coordinates change.
axes[0].scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", s=140)
axes[0].set(title="Input space", xlabel="x1", ylabel="x2")
axes[1].scatter(hidden[:, 0], hidden[:, 1], c=y.ravel(), cmap="coolwarm", s=140)
axes[1].set(title="Hidden space", xlabel="h1: any", ylabel="h2: both")
for axis in axes:
    axis.grid(alpha=0.3)
plt.show()
print("Measured payoff: the same four cases are now linearly readable.")


### That works. But how would a weight know which way to move?

The XOR construction proved that useful hidden features **exist**. We chose them. To understand how training could adjust weights, the next cell starts a **separate** tiny network with deliberately imperfect weights and one case `[1, 0]`, target `1`.

Follow its values before following the derivatives:

```text
[1,0] -> hidden pre-activations [0.9,-0.6]
      -> ReLU activations       [0.9, 0.0]
      -> 0.9*0.7 + 0*(-0.5) + 0.05 = output 0.68
      -> half squared error: 0.5*(0.68-1)² = loss 0.0512
```

A **pre-activation** (`z1`) is the weighted sum before the gate; an **activation** (`h1`) is the value after it. The half in this teaching loss makes its output derivative simply `output - target = -0.32`.

Now ask: “if I increase a weight a tiny amount, how does this loss change?” That local slope is its **gradient**. A negative gradient means a small increase should lower loss nearby; a positive one means a small decrease should. It is sensitivity, not a percentage of blame or a causal explanation of the dataset.

Backpropagation applies the **chain rule**: multiply sensitivities along a path, then add contributions where paths meet. For the first input-to-hidden weight:

```text
loss sensitivity at output        -0.32
output sensitivity to hidden 1   * 0.7
open ReLU's local slope          * 1
hidden sum sensitivity to weight * input A = 1
result                           -0.224
```

The second hidden gate is closed because its sum is -0.6, so its local slope is zero. Signal B is zero, so both of its input-weight gradients are zero for this case. The four-weight report must be:

```text
dW1 = [ -0.224   0 ]   rows: input A, input B
      [  0       0 ]   columns: hidden unit 1, hidden unit 2
```

> **Visual to add — A gradient stops at the closed gate**
>
> Draw the actual one-case values [1,0], pre-activations [0.9,-0.6], activations [0.9,0], output 0.68, and loss 0.0512. Trace -0.32 backward through weight 0.7 to -0.224; stop the other path at the closed ReLU. Show zero input B separately from a closed gate: two different reasons for zero gradients. Do not change weights along these arrows; computing gradients is not an optimizer update.
>
> **Proposed caption:** “Backpropagation reuses the forward values to report local sensitivity.”

In the cell, `.T` swaps matrix axes so each returned gradient has the same layout as its weight table. `dW2` measures output weights, `dh1` measures activations, and `dz1` is the report after applying the ReLU gates. The printed report is `dW1`, not every parameter's gradient.

In [ ]:
# -- One-case forward pass and manual responsibility report -------------------
# (1, 2) keeps the case axis; target=(1, 1) keeps the matching output axis.
x_case = np.array([[1.0, 0.0]])
target = np.array([[1.0]])
W1 = np.array([[0.8, -0.4], [0.3, 0.9]])
b1 = np.array([[0.1, -0.2]])
W2 = np.array([[0.7], [-0.5]])
b2 = np.array([[0.05]])

# (1, 2) @ (2, 2) + (1, 2) -> (1, 2): two hidden questions for this one case.
z1 = x_case @ W1 + b1
h1 = np.maximum(z1, 0.0)
# (1, 2) @ (2, 1) + (1, 1) -> (1, 1): combine those features into one score.
output = h1 @ W2 + b2
loss = 0.5 * (output - target) ** 2

d_output = output - target
# h1.T turns (1, 2) into (2, 1); @ (1, 1) -> (2, 1), one report per output weight.
dW2 = h1.T @ d_output
# (1, 1) @ W2.T=(1, 2) -> (1, 2): send the error back along each hidden-to-output link.
dh1 = d_output @ W2.T
# Two matching (1, 2) arrays: gate each hidden report without mixing the units.
dz1 = dh1 * (z1 > 0)
# x_case.T=(2, 1) @ (1, 2) -> (2, 2): pair every input with every hidden report,
# giving exactly the same input-by-hidden layout as W1.
dW1 = x_case.T @ dz1

print(f"Forward: x{x_case.shape} -> z{z1.shape} -> h{h1.shape} -> output{output.shape}")
# (1, 1) -> scalar for reporting this one-case loss; no case averaging is hidden here.
print(f"Loss={loss.item():.6f}")
print("Responsibility report dW1:")
print(dW1)


## 3 · Check the report by moving a weight

If the first weight really has slope `-0.224`, a small increase should lower loss. As a separately checked illustration, moving it from `0.80` to `0.81` changes output from `0.680` to `0.687` and loss from `0.0512` to `0.0489845`. That finite move is not executed by a training loop here.

The next cell uses a much smaller nudge, `epsilon = 0.00001`, on **each of the four W1 entries**. It renders once with that weight increased and once decreased, holding every other value fixed. Then it divides the loss difference by `2*epsilon`, the full distance between the two trials.

This is a **central finite difference**, an independent numerical estimate of the local slope. It costs two forward calculations per weight, so eight here. Backpropagation obtains the table by reusing one forward calculation.

The assertion requires the maximum absolute disagreement to be below `1e-7`. It checks this one smooth local case, not every input or parameter. Both pre-activations stay away from zero during the nudges; at ReLU's corner a two-sided difference need not match the code's chosen `z > 0` derivative convention.

In [ ]:
# -- Compare analytic and finite-difference gradients -------------------------
def loss_for_W1(candidate_W1):
    # Keep the audit's forward shapes identical: (1, 2) @ (2, 2) + (1, 2) -> (1, 2).
    candidate_z = x_case @ candidate_W1 + b1
    candidate_h = np.maximum(candidate_z, 0.0)
    # (1, 2) @ (2, 1) + (1, 1) -> (1, 1); item() below returns that one loss as a scalar.
    candidate_output = candidate_h @ W2 + b2
    return (0.5 * (candidate_output - target) ** 2).item()

epsilon = 1e-5
finite_difference = np.zeros_like(W1)
for row in range(W1.shape[0]):
    for column in range(W1.shape[1]):
        plus = W1.copy()
        minus = W1.copy()
        # Select one scalar in each (2, 2) copy, so the check isolates one weight's effect.
        plus[row, column] += epsilon
        minus[row, column] -= epsilon
        finite_difference[row, column] = (
            loss_for_W1(plus) - loss_for_W1(minus)
        ) / (2 * epsilon)

max_error = np.max(np.abs(dW1 - finite_difference))
print("Finite-difference report:")
print(finite_difference)
print(f"Maximum absolute disagreement: {max_error:.2e}")
assert max_error < 1e-7


## 4 · More units are not the same as better learning

We have a representation and a way to check a derivative. A few related terms are worth separating before they become a shopping list:

- **Width:** how many intermediate features a layer produces. Our hidden layer is two units wide. More units offer more features, not a promise that the features will help.
- **Depth:** how many transformations are composed. We used a hidden affine transformation, ReLU, and an output affine transformation. Repeating affine layers **without** a nonlinearity still collapses into one affine map; it would not fix XOR.
- **Normalization:** rescale values using statistics along a chosen axis. For an illustrative row `[2, 4]`, subtracting its mean `3` gives `[-1, 1]`. A real normalization layer also specifies spread, axes, and a numerical stabilizer. None is implemented here.
- **Dropout:** randomly zero some intermediate values during training. It discourages relying on a single route, but this notebook checks only its scaling, not any generalization benefit.

With dropout probability `0.4`, an activation of `1` is either dropped to `0` or kept and divided by `0.6`, becoming about `1.667`. Across repeated masks its expected value stays `0.4*0 + 0.6*(1/0.6) = 1`. Evaluation leaves it at `1` without dropping or rescaling.

> **Visual to add — Keep the average, not every light**
>
> Draw ten illustrative activation lamps of value 1. In a training panel switch four off and label six survivors 1/0.6; in evaluation keep all ten at 1. Add “one illustrative mask; the code samples 10,000 activations.” Do not turn this exact four-of-ten arrangement into a claim that every random mask drops precisely 40%, or show evaluation applying another 0.6 factor.
>
> **Proposed caption:** “Inverted dropout preserves the average in expectation, not each activation.”

The next cell samples 10,000 all-one activations. Its observed training mean should be near, not necessarily exactly, the evaluation mean of 1. It trains no model and does not measure uncertainty calibration.

In [ ]:
# -- Measure dropout's train/evaluation contract ------------------------------
rng = np.random.default_rng(5)
activation = np.ones((10_000,))
drop_probability = 0.4
mask = rng.random(activation.shape) >= drop_probability
# Matching (10_000,) arrays drop individual activations, not whole rows or feature groups.
training_activation = activation * mask / (1 - drop_probability)
evaluation_activation = activation

print(f"Training mean after inverted dropout: {training_activation.mean():.3f}")
print(f"Evaluation mean with no dropping: {evaluation_activation.mean():.3f}")
print("The scaling preserves expectation, not each individual activation.")


### Your turn · Close one route, then read what disappears

The next cell shifts the first hidden pre-activation from `0.9` by `YOUR_BIAS_SHIFT`. At the supplied `-1.0`, it becomes `-0.1` and closes the gate. The second was already closed.

Keep the earlier upstream sensitivity `dh1` fixed and watch the gate multiply it by zero. This is a **local gate demonstration**, not a fresh forward/backward pass through the modified network: the cell does not recompute its output or loss. Try a smaller negative shift to keep the first gate open and compare.

In [ ]:
# -- CHANGE THIS: make one hidden pre-activation negative ----------------------
YOUR_BIAS_SHIFT = -1.0
changed_z = z1.copy()
# Select one scalar in (1, 2): close the first hidden route without touching the second.
changed_z[0, 0] += YOUR_BIAS_SHIFT
changed_gate = (changed_z > 0).astype(int)
# (1, 2) * (1, 2) applies one gate per hidden report, preserving the original unit order.
changed_gradient = dh1 * changed_gate
# Each (1, 2) -> (2,) flatten below is display-only; keep case/unit axes for calculations.
print(f"Changed pre-activation: {changed_z.ravel().tolist()}")
print(f"ReLU gates: {changed_gate.ravel().tolist()}")
print(f"Responsibility reaching the hidden units: {changed_gradient.ravel().tolist()}")


### What this little network did not learn

The hand-set XOR weights express the four-case rule perfectly. That establishes capacity, not successful optimization. The separate one-case gradient audit establishes a local derivative, not accuracy on new listings.

A zero gradient can come from an inactive input or a closed gate; it does not prove a weight is useless on every case. A correct nonzero gradient only suggests a local direction. Step size and repeated measurement still matter.

The next stage will give parameters and gradients persistent homes in a framework and let an optimizer apply updates. This chapter stops before that loop so you can see which part computes the guidance and which part must act on it.

## The short version

- Fair tests cannot rescue a rule that cannot express the labels.
- Hidden features change the coordinates; ReLU prevents the whole network collapsing into one affine rule.
- Our XOR features were chosen by hand. They were not discovered by training.
- A gradient says how a tiny parameter change affects loss locally. It is not the parameter update itself.
- Chain rule combines local sensitivities. Closed gates and zero inputs can stop individual contributions.
- Numerical nudges check derivatives; inverted dropout checks an expectation. Neither proves generalization.

**Where we stopped**

- **Built and checked:** the linear grid search, four-case hidden representation, one-case manual backward pass, a four-entry W1 finite-difference audit, and dropout scaling.
- **Talked through:** width, depth, normalization, ReLU's corner, and gradient direction.
- **Left for Stage 02:** repeated optimizer updates, autograd APIs, initialization, and held-out training comparisons.

For an optional audible version, [Melodyne](optional/02-melodyne-backprop-synthesizer.ipynb) keeps the chain rule but replaces the neural network with four adjustable synthesizer controls.

The required next step is [The Deep-Learning Map](../../02-pytorch-foundations/01-deep-learning-map/01-deep-learning-map.ipynb): the same distinction between forward calculation, stored gradients, and a separate optimizer, now in PyTorch.